# exp021_distance_weighted_inference_postprocess inference

Final inference for the exp020 selected distance-weighted LightGBM profile with the configured postprocess method.


## Contents

1. Setup and configuration
2. Selected inference candidate
3. Fit weighted model and generate submission
4. Artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from distance_weighted_inference_postprocess import (
    experiment_config,
    generate_weighted_submission,
    selected_training_variant,
    train_files,
    test_files,
)
from settings import EXPERIMENT_NAME, ExperimentPaths

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = experiment_config()

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Test data:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)


## 2. Selected Inference Candidate


In [ ]:
variant = selected_training_variant(config)
model_files = train_files(paths, max_wells=MAX_WELLS if DEBUG else MAX_WELLS)
infer_files = test_files(paths)
print("Selected training variant:", variant["name"])
print("Weight profile:", variant.get("weight_profile"))
print("Selected postprocess:", config["postprocess"].get("selected_method"))
print("Train wells:", len(model_files))
print("Test wells:", len(infer_files))
print("Final row cap:", config["model"]["training"].get("max_train_rows_final"))
print("Rows per well cap:", config["model"]["training"].get("max_train_rows_per_well"))


## 3. Fit Weighted Model And Generate Submission


In [ ]:
summary = generate_weighted_submission(
    paths,
    config,
    max_wells=MAX_WELLS if DEBUG else MAX_WELLS,
)
print(json.dumps(summary, indent=2, sort_keys=True))


## 4. Artifacts


In [ ]:
for path in sorted(paths.artifacts_dir.glob("*")):
    if path.is_file():
        print(path.name, path.stat().st_size)
print("Submission exists:", paths.submission_path.exists())
if paths.submission_path.exists():
    print(paths.submission_path)
    print(paths.submission_path.stat().st_size)
